In [13]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/lukajincharadze/pandas-data/noc_regions.csv
/kaggle/input/datasets/lukajincharadze/pandas-data/olympics-data.xlsx
/kaggle/input/datasets/lukajincharadze/pandas-data/results.parquet
/kaggle/input/datasets/lukajincharadze/pandas-data/bios.csv
/kaggle/input/datasets/lukajincharadze/pandas-data/coffee.csv
/kaggle/input/datasets/lukajincharadze/pandas-data/results.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


# DagsHub/Mlflow Initialization

In [25]:
%pip install -q dagshub mlflow

Note: you may need to restart the kernel to use updated packages.


In [15]:
import dagshub
dagshub.init(repo_owner='skoba23', repo_name='ML_Assignment_1', mlflow=True)

Initialized MLflow to track repo "skoba23/ML_Assignment_1"

Repository skoba23/ML_Assignment_1 initialized!

In [16]:
import mlflow
with mlflow.start_run():
  # Your training code here...
  mlflow.log_metric('accuracy', 42)
  mlflow.log_param('Param name', 'Value')

🏃 View run masked-whale-877 at: https://dagshub.com/skoba23/ML_Assignment_1.mlflow/#/experiments/0/runs/d4c14e08965344af98873365f3e25d6a
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_1.mlflow/#/experiments/0


In [17]:
mlflow.autolog()


2026/04/11 13:31:01 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/04/11 13:31:01 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.


# Cleaning Data / Feature Engineering

In [18]:

df_train = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv")
df_test = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv")

print(df_train.shape)
df_train.head()


(1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [19]:
df_train.isnull().sum()[df_train.isnull().sum() > 0]

LotFrontage      259
Alley           1369
MasVnrType       872
MasVnrArea         8
BsmtQual          37
BsmtCond          37
BsmtExposure      38
BsmtFinType1      37
BsmtFinType2      38
Electrical         1
FireplaceQu      690
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
PoolQC          1453
Fence           1179
MiscFeature     1406
dtype: int64

In [20]:
num_cols = df_train.select_dtypes(include=np.number).columns
df_train[num_cols] = df_train[num_cols].fillna(df_train[num_cols].median())

In [21]:
cat_cols = df_train.select_dtypes(include = 'object').columns
df_train[cat_cols] = df_train[cat_cols].fillna("None")

### One-Hot Encoding

In [22]:
df_train = pd.get_dummies(df_train, columns = cat_cols)
df_test = pd.get_dummies(df_test, columns = cat_cols)

df_train, df_test = df_train.align(df_test, join = 'left', axis = 1, fill_value = 0)

In [23]:
df_train.head()

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
0,1,60,65.0,8450,7,5,2003,2003,196.0,706,...,False,False,False,True,False,False,False,False,True,False
1,2,20,80.0,9600,6,8,1976,1976,0.0,978,...,False,False,False,True,False,False,False,False,True,False
2,3,60,68.0,11250,7,5,2001,2002,162.0,486,...,False,False,False,True,False,False,False,False,True,False
3,4,70,60.0,9550,7,5,1915,1970,0.0,216,...,False,False,False,True,True,False,False,False,False,False
4,5,60,84.0,14260,8,5,2000,2000,350.0,655,...,False,False,False,True,False,False,False,False,True,False


In [29]:
df_train["TotalSF"] = df_train["TotalBsmtSF"] + df_train["1stFlrSF"] + df_train["2ndFlrSF"]
df_train["HouseAge"] = df_train["YrSold"] - df_train["YearBuilt"]
df_train["YearSinceRemodel"] = df_train["YrSold"] - df_train["YearRemodAdd"]

In [31]:
df_train.shape

(1460, 308)

In [32]:
print(df_train.isnull().sum()[df_train.isnull().sum() > 0])
#there is no Nan now.

Series([], dtype: int64)


In [34]:
correlation = df_train.corr()['SalePrice'].abs().sort_values(ascending=False)
print(correlation.head(20))

SalePrice           1.000000
OverallQual         0.790982
TotalSF             0.782260
GrLivArea           0.708624
GarageCars          0.640409
GarageArea          0.623431
TotalBsmtSF         0.613581
1stFlrSF            0.605852
ExterQual_TA        0.589044
FullBath            0.560664
BsmtQual_Ex         0.553105
TotRmsAbvGrd        0.533723
HouseAge            0.523350
YearBuilt           0.522897
KitchenQual_TA      0.519298
YearSinceRemodel    0.509079
YearRemodAdd        0.507101
KitchenQual_Ex      0.504094
Foundation_PConc    0.497734
MasVnrArea          0.472614
Name: SalePrice, dtype: float64


# Feature Selection

In [40]:
threshold = 0.05
good_features = correlation[correlation > threshold].index.tolist()

df_train = df_train[good_features]
print(f"there is {len(good_features)} columns left")

there is 196 columns left


In [41]:
df_train.head()

,SalePrice,OverallQual,TotalSF,GrLivArea,GarageCars,GarageArea,TotalBsmtSF,1stFlrSF,ExterQual_TA,FullBath,...,BsmtFinType2_BLQ,Neighborhood_ClearCr,BsmtCond_Po,Exterior2nd_Plywood,Exterior1st_WdShing,Exterior1st_BrkComm,Fence_MnWw,LandSlope_Gtl,SaleCondition_AdjLand,ExterCond_Gd
0,208500,7,2566,1710,2,548,856,856,False,2,...,False,False,False,False,False,False,False,True,False,False
1,181500,6,2524,1262,2,460,1262,1262,True,2,...,False,False,False,False,False,False,False,True,False,False
2,223500,7,2706,1786,2,608,920,920,False,2,...,False,False,False,False,False,False,False,True,False,False
3,140000,7,2473,1717,3,642,756,961,True,1,...,False,False,False,False,False,False,False,True,False,False
4,250000,8,3343,2198,3,836,1145,1145,False,2,...,False,False,False,False,False,False,False,True,False,False


# Training

In [44]:
Y_train = df_train['SalePrice']
X_train = df_train.drop('SalePrice', axis=1)

print(X_train.shape)
print(Y_train.shape)

(1460, 195)
(1460,)


In [49]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import numpy as np

X_tr, X_val, Y_tr, Y_val = train_test_split(X_train, Y_train, test_size=0.2, random_state=42)


### Linear Regression

In [57]:
with mlflow.start_run(run_name = "LinearRegressionBaseline"):
    model = LinearRegression()
    model.fit(X_tr, Y_tr)

    predict = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(Y_val, predict))

    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_metric("rmse", rmse)
    mlflow.sklearn.log_model(model, "model")

    print(f"Linear Regression RMSE: {rmse:.2f}")

2026/04/11 15:50:27 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/04/11 15:50:27 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.12/dist-packages/mlflow/types/utils

Linear Regression RMSE: 29661.61
🏃 View run LinearRegressionBaseline at: https://dagshub.com/skoba23/ML_Assignment_1.mlflow/#/experiments/0/runs/f2803359d57d4cd789237350528b6f30
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_1.mlflow/#/experiments/0


### Random Forest

In [59]:
with mlflow.start_run(run_name = "RandomForestBaseline"):
    model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
    model_rf.fit(X_tr, Y_tr)

    predict_rf = model_rf.predict(X_val)
    rmse_rf = np.sqrt(mean_squared_error(Y_val, predict_rf))

    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_metric("rmse", rmse_rf)
    mlflow.sklearn.log_model(model_rf, "model")

    print(f"Random Forest RMSE: {rmse_rf:.2f}")

2026/04/11 15:51:44 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/04/11 15:51:48 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.12/dist-packages/mlflow/types/utils

Random Forest RMSE: 30003.39
🏃 View run RandomForestBaseline at: https://dagshub.com/skoba23/ML_Assignment_1.mlflow/#/experiments/0/runs/868fd5995dfc46f1a40b28db21912cbd
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_1.mlflow/#/experiments/0


In [53]:
print(Y_train.unique()[:10])
print(Y_train.dtype)

[208500 181500 223500 140000 250000 143000 307000 200000 129900 118000]
int64


In [60]:
print(f"Linear Regression RMSE: {np.sqrt(mean_squared_error(Y_val, predict)):.2f}")
print(f"Random Forest RMSE: {np.sqrt(mean_squared_error(Y_val, predict_rf)):.2f}")

Linear Regression RMSE: 29661.61
Random Forest RMSE: 30003.39


### XGBoost

In [61]:
with mlflow.start_run(run_name = "XGBoostBaseline"):
    model_XGB = XGBRegressor(n_estimators=100, learning_rate = 0.1, max_depth = 5, random_state=42)
    model_XGB.fit(X_tr, Y_tr)

    predict_XGB = model_XGB.predict(X_val)
    rmse_XGB = np.sqrt(mean_squared_error(Y_val, predict_XGB))

    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("max_depth", 5)
    mlflow.log_metric("rmse", rmse_XGB)
    mlflow.sklearn.log_model(model_XGB, "model")

    print(f"XGBoost RMSE: {rmse_XGB:.2f}")

2026/04/11 16:04:25 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/04/11 16:04:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/11 16:04:35 WARNING mlflow.utils.autologging_u

XGBoost RMSE: 26139.49
🏃 View run XGBoostBaseline at: https://dagshub.com/skoba23/ML_Assignment_1.mlflow/#/experiments/0/runs/334051d78ddb48c18697002270d51699
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_1.mlflow/#/experiments/0


### XGBoost with hyperparameter

In [63]:
with mlflow.start_run(run_name = "XGBoostUpgraded"):
    model_XGB_2 = XGBRegressor(n_estimators=500, 
                               learning_rate = 0.05, 
                               max_depth = 4,
                               subsample = 0.8,
                               colsample_bytree = 0.8,
                               random_state=42)
    model_XGB_2.fit(X_tr, Y_tr)

    predict_XGB_2 = model_XGB_2.predict(X_val)
    rmse_XGB_2 = np.sqrt(mean_squared_error(Y_val, predict_XGB_2))

    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("n_estimators", 500)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("max_depth", 4)
    mlflow.log_metric("rmse", rmse_XGB_2)
    mlflow.sklearn.log_model(model_XGB_2, "model")

    print(f"Upgraded XGBoost RMSE: {rmse_XGB_2:.2f}")

2026/04/11 16:10:50 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/04/11 16:10:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/11 16:10:59 WARNING mlflow.utils.autologging_u

Upgraded XGBoost RMSE: 25063.18
🏃 View run XGBoostUpgraded at: https://dagshub.com/skoba23/ML_Assignment_1.mlflow/#/experiments/0/runs/338b6a3e831446beaa5fd28050d5e751
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_1.mlflow/#/experiments/0


In [64]:
with mlflow.start_run(run_name = "XGBoostUpgraded_v2"):
    model_XGB_3 = XGBRegressor(n_estimators=1000, 
                               learning_rate = 0.01, 
                               max_depth = 4,
                               subsample = 0.8,
                               colsample_bytree = 0.8,
                               min_child_weight=3,
                               random_state=42)
    model_XGB_3.fit(X_tr, Y_tr)

    predict_XGB_3 = model_XGB_3.predict(X_val)
    rmse_XGB_3 = np.sqrt(mean_squared_error(Y_val, predict_XGB_3))

    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("n_estimators", 1000)
    mlflow.log_param("learning_rate", 0.01)
    mlflow.log_param("max_depth", 4)
    mlflow.log_param("min_child_weight", 3)
    mlflow.log_metric("rmse", rmse_XGB_3)
    mlflow.sklearn.log_model(model_XGB_3, "model")

    print(f"Upgraded XGBoost v2 RMSE: {rmse_XGB_3:.2f}")

2026/04/11 16:39:00 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/04/11 16:39:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/11 16:39:10 WARNING mlflow.utils.autologging_u

Upgraded XGBoost v2 RMSE: 25302.92
🏃 View run XGBoostUpgraded_v2 at: https://dagshub.com/skoba23/ML_Assignment_1.mlflow/#/experiments/0/runs/01b3600ae96649ddbe9f482ad714fe98
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_1.mlflow/#/experiments/0


# Saving to Model Registry

In [66]:
with mlflow.start_run(run_name = "XGB_Best_Model"):
    mlflow.sklearn.log_model(model_XGB_2, artifact_path="model", registered_model_name="House-Prices-XGBoost")
    mlflow.log_metric("rmse", rmse_XGB_2)
    print("Model is Saved to Registry!")
 

2026/04/11 16:45:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/11 16:45:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'House-Prices-XGBoost'.
2026/04/11 16:46:05 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: House-Prices-XGBoost, version 1
Created version '1' of model 'House-Prices-XGBoost'.


Model is Saved to Registry!
🏃 View run XGB_Best_Model at: https://dagshub.com/skoba23/ML_Assignment_1.mlflow/#/experiments/0/runs/e8cfb88db22b422f94aa98ec0ec5abe0
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_1.mlflow/#/experiments/0
